# Can we predict next-month churn?
Target: `churn_next_month` (1 = account cancels in the month after the snapshot). One row per account-month.

Plan:
1. Inspect the data, including a **leakage check** on every column.
2. Split honestly: by time (train on the past, test on the future) and with account-grouped CV.
3. Compare a base-rate baseline, logistic regression and gradient boosting.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, brier_score_loss
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
df = pd.read_csv('data.csv')
print(df.shape, '| accounts:', df.account_ref.nunique(), '| months:', df.snapshot_month.min(), '->', df.snapshot_month.max())
print('missing values:', int(df.isna().sum().sum()), '| duplicate rows:', int(df.duplicated().sum()))
print('churn rate: %.3f' % df.churn_next_month.mean())
df.describe().T

## 1. Leakage check
`lifetime_months` is described as *total months the account was active, from the CRM export of 2026-01-05*. That is information from after the snapshot. If an account cancels next month, its lifetime ends at the snapshot, so it should equal `tenure_months`. Test that.

In [ ]:
gap = df.lifetime_months - df.tenure_months
print(gap.groupby(df.churn_next_month).describe())
print('\nlifetime == tenure  ->  churn rate: %.3f' % df.churn_next_month[gap == 0].mean())
print('lifetime >  tenure  ->  churn rate: %.3f' % df.churn_next_month[gap > 0].mean())
print('AUC of (lifetime == tenure) alone: %.3f' % roc_auc_score(df.churn_next_month, (gap == 0).astype(int)))

`lifetime_months == tenure_months` happens **only** for churning rows: it encodes the label from the future. It would not exist at prediction time, so **`lifetime_months` is excluded**. The rest of the notebook uses only columns known at the end of the snapshot month. `account_ref` is an identifier and is used only for grouping.

## 2. Exploration

In [ ]:
num = ['seats','monthly_fee','tenure_months','usage_hours','support_tickets']
fig, ax = plt.subplots(2, 3, figsize=(14, 7)); ax = ax.ravel()
for a, col in zip(ax, num):
    q = pd.qcut(df[col], 8, duplicates='drop') if df[col].nunique() > 8 else df[col]
    r = df.groupby(q, observed=True).churn_next_month.mean()
    a.bar(range(len(r)), r.values, color='#4477aa'); a.set_xticks(range(len(r)))
    a.set_xticklabels([str(i) for i in r.index], rotation=45, ha='right', fontsize=7)
    a.axhline(df.churn_next_month.mean(), color='k', ls='--', lw=1); a.set_title(f'churn rate by {col}')
r = df.groupby('plan').churn_next_month.mean(); ax[5].bar(r.index, r.values, color='#4477aa'); ax[5].set_title('churn rate by plan')
ax[5].axhline(df.churn_next_month.mean(), color='k', ls='--', lw=1)
plt.tight_layout(); plt.show()
print(df.groupby('snapshot_month').churn_next_month.agg(['mean','size']).T.round(3).to_string())

## 3. Features and split
Features: the raw snapshot columns, fee per seat, and **account history built only from earlier rows** (previous usage, change in usage, rolling mean of usage) and the calendar month (churn is strongly seasonal: Nov-Jan is far above the rest of the year). First snapshots have no history (NaN, handled by the boosting model; filled for logistic regression).

Split: the last 4 snapshot months are the test set (the model is trained on the past and judged on the future, as in deployment). Accounts appear in many months, so I also run grouped CV by account for a second opinion. Note that accounts in the temporal test set can also appear in training; that is also true in deployment.

In [ ]:
df = df.sort_values(['account_ref','snapshot_month']).reset_index(drop=True)
g = df.groupby('account_ref')
df['fee_per_seat'] = df.monthly_fee / df.seats
df['usage_prev'] = g.usage_hours.shift(1)
df['usage_delta'] = df.usage_hours - df.usage_prev
df['usage_roll3'] = g.usage_hours.transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df['cal_month'] = df.snapshot_month.str[-2:].astype(int)
df['tickets_prev'] = g.support_tickets.shift(1)
df['usage_vs_roll'] = df.usage_hours - df.usage_roll3
cat = ['plan']
numf = ['seats','monthly_fee','fee_per_seat','tenure_months','usage_hours','support_tickets',
        'usage_prev','usage_delta','usage_roll3','usage_vs_roll','tickets_prev','cal_month']
X, y, groups = df[cat+numf], df.churn_next_month, df.account_ref
months = sorted(df.snapshot_month.unique()); cut = months[-4]
test = (df.snapshot_month >= cut).values
print('train through', months[-5], '| test from', cut, '| train n=%d (churn %.3f) test n=%d (churn %.3f)' % (
    (~test).sum(), y[~test].mean(), test.sum(), y[test].mean()))

In [ ]:
def prep(scale):
    nums = make_pipeline(*( [__import__('sklearn.impute', fromlist=['x']).SimpleImputer(strategy='median')] + ([StandardScaler()] if scale else [])))
    return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), cat), ('n', nums, numf)])
models = {
 'baseline (base rate)': DummyClassifier(strategy='prior'),
 'calendar month only (seasonality)': make_pipeline(ColumnTransformer([('m', OneHotEncoder(handle_unknown='ignore'), ['cal_month'])]), LogisticRegression(max_iter=1000)),
 'logistic regression': make_pipeline(prep(True), LogisticRegression(max_iter=2000, C=1.0)),
 'hist gradient boosting': make_pipeline(prep(False), HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, l2_regularization=1.0, random_state=0)),
}
rows, probs = [], {}
for name, m in models.items():
    m.fit(X[~test], y[~test]); p = m.predict_proba(X[test])[:, 1]; probs[name] = p
    cvp = cross_val_predict(m, X, y, groups=groups, cv=GroupKFold(5), method='predict_proba')[:, 1]
    rows.append({'model': name, 'test ROC-AUC': roc_auc_score(y[test], p), 'test PR-AUC': average_precision_score(y[test], p),
                 'test Brier': brier_score_loss(y[test], p), 'groupCV ROC-AUC': roc_auc_score(y, cvp), 'groupCV PR-AUC': average_precision_score(y, cvp)})
res = pd.DataFrame(rows).set_index('model').round(3); print('test base rate (= PR-AUC of a random model): %.3f' % y[test].mean()); res

## 4. What does the model get us?

In [ ]:
yt = y[test].values; pb = probs['hist gradient boosting']; pl = probs['logistic regression']
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for n, p in [('logistic', pl), ('boosting', pb)]:
    pr, rc, _ = precision_recall_curve(yt, p); ax[0].plot(rc, pr, label=n)
ax[0].axhline(yt.mean(), color='k', ls='--', label='base rate'); ax[0].set_xlabel('recall'); ax[0].set_ylabel('precision'); ax[0].legend(); ax[0].set_title('Precision-recall (test)')
d = pd.DataFrame({'y': yt, 'p': pb}); d['decile'] = pd.qcut(d.p.rank(method='first'), 10, labels=False) + 1
cal = d.groupby('decile').agg(pred=('p','mean'), actual=('y','mean'))
ax[1].plot(cal.pred, cal.actual, 'o-'); ax[1].plot([0, cal.values.max()], [0, cal.values.max()], 'k--'); ax[1].set_xlabel('mean predicted'); ax[1].set_ylabel('observed churn'); ax[1].set_title('Calibration by decile (boosting)')
plt.tight_layout(); plt.show()
top = d[d.decile >= 9]
print('Top 20%% of accounts by risk: churn rate %.3f (lift %.1fx), captures %.0f%% of all churners' % (top.y.mean(), top.y.mean()/yt.mean(), 100*top.y.sum()/yt.sum()))

In [ ]:
m = models['hist gradient boosting']
pi = permutation_importance(m, X[test], y[test], scoring='roc_auc', n_repeats=10, random_state=0)
pd.Series(pi.importances_mean, index=X.columns).sort_values().plot.barh(figsize=(7, 4), color='#4477aa', title='Permutation importance (drop in test ROC-AUC)')
plt.tight_layout(); plt.show()

## 5. Sensitivity: what if we had (wrongly) kept `lifetime_months`?

In [ ]:
Xl = X.assign(lifetime_months=df.lifetime_months); numf_l = numf + ['lifetime_months']
ml = make_pipeline(ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), cat), ('n', 'passthrough', numf_l)]),
                   HistGradientBoostingClassifier(max_depth=4, random_state=0)).fit(Xl[~test], y[~test])
print('boosting WITH leaked lifetime_months: test ROC-AUC = %.3f  (vs %.3f honest)' % (
    roc_auc_score(yt, ml.predict_proba(Xl[test])[:, 1]), res.loc['hist gradient boosting','test ROC-AUC']))

## Conclusions

**Can we predict churn? Partly: it ranks risk better than chance, but modestly, and the numbers need care.**

- **Leakage:** `lifetime_months` equals `tenure_months` for exactly the accounts that churn next month (a post-hoc CRM field). Using it gives ROC-AUC ~0.99, which is meaningless. It was excluded; any model using it must not be trusted.
- **Honest performance (train on the past, test on Aug-Nov 2025):** ROC-AUC ~0.69-0.71 and PR-AUC ~0.20-0.24 vs a 0.093 base rate. The top 20% highest-risk accounts churn at about 2x the average rate and contain ~40% of churners. Useful for prioritising outreach, not for confident individual predictions.
- **Seasonality dominates:** churn is ~50% in Dec/Jan and ~7% in spring/summer. Account-level signals (usage, tickets, plan, seats, tenure) are weak on their own.
- **Group CV is optimistic here.** It shuffles months, so the boosting model sees the same winter periods in training and test (ROC-AUC 0.80, PR-AUC 0.48). The temporal test, which has no winter in it, is the better guide to a new period. Calendar month alone scores 0.72 in group CV but 0.55 on the temporal test.
- **Calibration:** on the temporal test the models' Brier score is *worse* than the base rate, because the training period contains the high-churn winter. Probabilities are over-stated for off-season months and should be recalibrated (or seasonal-adjusted) before use as absolute rates.
- **Limits and next steps:** only ~2 winters of data, so seasonal effects cannot be validated; a test that includes Dec/Jan would be the real check (wait for those months or hold out winter 2024-25). Better signals (billing failures, contract renewal dates, NPS, feature usage) would likely help more than model tuning. Results come from one split with no tuning, so differences between logistic regression and boosting are within noise.